# EcoShield AI — Ağır Model Karşılaştırması

Bu notebook ortak split üzerinde dört ağır model adayını karşılaştırır:

1. Heavy Random Forest
2. XGBoost
3. Heavy LightGBM
4. Heavy CatBoost

Karşılaştırma yalnızca train ve validation splitlerinde yapılır. Test spliti belleğe yüklenmez. Threshold tuning ve kesin model seçimi bu görevde yapılmaz; `0.50` eşiği yalnızca tanısal metrik üretmek için kullanılır.

Bellek kullanımını sınırlamak için preprocessing profilleri sırayla yüklenir ve ilgili modeller tamamlanınca serbest bırakılır.

In [1]:
%pip install xgboost

  Using cached xgboost-3.2.0-py3-none-win_amd64.whl.metadata (2.1 kB)
Using cached xgboost-3.2.0-py3-none-win_amd64.whl (101.7 MB)
Note: you may need to restart the kernel to use updated packages.


## 1. Ayarlar ve paket kontrolü

In [2]:
QUICK_MODE = False
RANDOM_STATE = 42
DIAGNOSTIC_THRESHOLD = 0.50
INFERENCE_SAMPLE_SIZE = 10_000
INFERENCE_REPEATS = 5
PREFERRED_DEVICE = "GPU"

import importlib.util
import sys
from pathlib import Path

required = {
    "catboost": "catboost", "joblib": "joblib", "lightgbm": "lightgbm",
    "numpy": "numpy", "pandas": "pandas", "psutil": "psutil",
    "scipy": "scipy", "sklearn": "scikit-learn", "tqdm": "tqdm",
    "xgboost": "xgboost",
}
missing = [pip for module, pip in required.items() if importlib.util.find_spec(module) is None]
if missing:
    raise ModuleNotFoundError(
        "Eksik paketler: " + ", ".join(missing)
        + ". Önce aktif torchcuda ortamına kurun."
    )

print("Ayarlar hazır.")
print("Tercih edilen cihaz:", PREFERRED_DEVICE)
print("Test spliti bu notebookta yüklenmeyecek.")

Ayarlar hazır.
Tercih edilen cihaz: GPU
Test spliti bu notebookta yüklenmeyecek.


## 2. Importlar, proje yolları ve ortak yardımcılar

In [3]:
import gc
import json
import os
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import psutil
from catboost import CatBoostClassifier
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from tqdm.auto import tqdm
import xgboost as xgb
from xgboost import XGBClassifier

warnings.filterwarnings("default")

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    candidate = NOTEBOOK_DIR / "notebooks"
    if (candidate / "common_preprocessing.py").exists():
        NOTEBOOK_DIR = candidate
if not (NOTEBOOK_DIR / "common_preprocessing.py").exists():
    raise FileNotFoundError("notebooks/common_preprocessing.py bulunamadı.")
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from common_preprocessing import (
    build_project_paths,
    find_project_root,
    get_or_create_profile_cache,
)

PROJECT_ROOT = find_project_root(Path.cwd())
PATHS = build_project_paths(PROJECT_ROOT)
MODEL_DIR = PROJECT_ROOT / "models" / "heavy"
PREDICTIONS_DIR = PATHS.outputs / "predictions"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

print("Proje kökü:", PROJECT_ROOT)
print("XGBoost:", xgb.__version__)
print("LightGBM:", lgb.__version__)

Proje kökü: C:\Users\pc\Desktop\YZTA-Bootcamp-2026
XGBoost: 3.2.0
LightGBM: 4.7.0


c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Ortak değerlendirme fonksiyonları

Bütün modeller için aynı validation metrikleri, eğitim süresi, inference süresi, RAM değişimi ve model dosya boyutu kaydedilir.

In [4]:
def ram_gb():
    return psutil.Process(os.getpid()).memory_info().rss / (1024 ** 3)


def select_rows(X, row_count):
    row_count = min(row_count, X.shape[0])
    if hasattr(X, "iloc"):
        return X.iloc[:row_count]
    return X[:row_count]


def calculate_validation_metrics(y_true, probabilities, threshold=DIAGNOSTIC_THRESHOLD):
    predictions = (np.asarray(probabilities) >= threshold).astype(np.int8)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "diagnostic_threshold": float(threshold),
        "precision": float(precision_score(y_true, predictions, zero_division=0)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "pr_auc": float(average_precision_score(y_true, probabilities)),
        "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
        "positive_prediction_rate_at_050": float(predictions.mean()),
    }


def measure_inference(model, X_validation):
    sample = select_rows(X_validation, INFERENCE_SAMPLE_SIZE)
    _ = model.predict_proba(sample)[:, 1]
    durations = []
    for _ in tqdm(range(INFERENCE_REPEATS), desc="10K inference", leave=False):
        started = time.perf_counter()
        _ = model.predict_proba(sample)[:, 1]
        durations.append(time.perf_counter() - started)
    return float(np.mean(durations)), float(np.std(durations)), sample.shape[0]


def finish_evaluation(
    *, model_name, model, data, probabilities, training_seconds,
    validation_inference_seconds, model_path, device,
    imbalance_method, ram_before, ram_after,
):
    inference_mean, inference_std, sample_size = measure_inference(
        model, data["X_validation"]
    )
    print(f"Model kaydediliyor: {model_path.name}")
    save_started = time.perf_counter()
    joblib.dump(model, model_path, compress=3)
    print(
        f"Model kaydı tamamlandı: {time.perf_counter() - save_started:.1f} sn "
        f"| {model_path.stat().st_size / (1024 ** 2):.2f} MB"
    )
    metrics = calculate_validation_metrics(data["y_validation"], probabilities)
    metrics.update({
        "model_name": model_name,
        "model_role": "heavy",
        "device": device,
        "split_version": "common_v2",
        "preprocessing_version": "lazy_cache_v1",
        "imbalance_method": imbalance_method,
        "threshold_selected": False,
        "training_seconds": float(training_seconds),
        "validation_inference_seconds": float(validation_inference_seconds),
        "inference_seconds_10k_mean": inference_mean,
        "inference_seconds_10k_std": inference_std,
        "inference_sample_rows": sample_size,
        "model_size_mb": model_path.stat().st_size / (1024 ** 2),
        "ram_before_gb": float(ram_before),
        "ram_after_fit_gb": float(ram_after),
    })
    print(pd.Series(metrics))
    return metrics

## 4. Sklearn-tree sparse cache'ini yükleme

Heavy Random Forest ve XGBoost aynı train-fitted sparse cache'i paylaşır. Test cache'i yüklenmez.

In [5]:
tree_data = get_or_create_profile_cache(
    "random_forest", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=False,
)
assert "X_test" not in tree_data

validation_ids = tree_data["id_validation"].copy()
validation_target = tree_data["y_validation"].copy()
print("Tree train:", tree_data["X_train"].shape)
print("Tree validation:", tree_data["X_validation"].shape)
print("RAM:", f"{ram_gb():.2f} GB")

sklearn_tree cache hazır; yeniden preprocessing yapılmayacak.
Tree train: (413378, 4068)
Tree validation: (88581, 4068)
RAM: 1.39 GB


## 5. Heavy Random Forest

In [6]:
heavy_rf_model = RandomForestClassifier(
    n_estimators=50,
    max_depth=None,
    min_samples_leaf=10,
    max_features="sqrt",
    class_weight="balanced_subsample",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    warm_start=True,
    verbose=0,
)

print("\nHEAVY RANDOM FOREST EĞİTİMİ")
ram_before = ram_gb()
started = time.perf_counter()
for tree_count in tqdm(
    range(50, 301, 50),
    desc="Heavy RF ağaçları",
    unit=" aşama",
):
    heavy_rf_model.set_params(n_estimators=tree_count)
    heavy_rf_model.fit(tree_data["X_train"], tree_data["y_train"])
    print(f"Tamamlanan ağaç: {tree_count}/300 | RAM: {ram_gb():.2f} GB")
heavy_rf_training_seconds = time.perf_counter() - started
ram_after = ram_gb()
started = time.perf_counter()
heavy_rf_probabilities = heavy_rf_model.predict_proba(
    tree_data["X_validation"]
)[:, 1]
heavy_rf_validation_inference = time.perf_counter() - started

heavy_rf_metrics = finish_evaluation(
    model_name="HeavyRandomForest",
    model=heavy_rf_model,
    data=tree_data,
    probabilities=heavy_rf_probabilities,
    training_seconds=heavy_rf_training_seconds,
    validation_inference_seconds=heavy_rf_validation_inference,
    model_path=MODEL_DIR / "heavy_random_forest.joblib",
    device="CPU",
    imbalance_method="balanced_subsample",
    ram_before=ram_before,
    ram_after=ram_after,
)


HEAVY RANDOM FOREST EĞİTİMİ


Heavy RF ağaçları:   0%|          | 0/6 [00:00<?, ? aşama/s]c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\sklearn\ensemble\_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(
Heavy RF ağaçları:  17%|█▋        | 1/6 [00:44<03:40, 44.11s/ aşama]

Tamamlanan ağaç: 50/300 | RAM: 1.42 GB


c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\sklearn\ensemble\_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(
Heavy RF ağaçları:  33%|███▎      | 2/6 [01:27<02:54, 43.71s/ aşama]

Tamamlanan ağaç: 100/300 | RAM: 1.45 GB


...

c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\sklearn\ensemble\_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(
Heavy RF ağaçları:  83%|████████▎ | 5/6 [03:36<00:43, 43.09s/ aşama]

Tamamlanan ağaç: 250/300 | RAM: 1.54 GB


c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\sklearn\ensemble\_forest.py:843: UserWarning: class_weight presets "balanced" or "balanced_subsample" are not recommended for warm_start if the fitted data differs from the full dataset. In order to use "balanced" weights, use compute_class_weight ("balanced", classes, y). In place of y you can use a large enough sample of the full training set target to properly estimate the class frequency distributions. Pass the resulting weights as the class_weight parameter.
  warn(
Heavy RF ağaçları: 100%|██████████| 6/6 [04:19<00:00, 43.23s/ aşama]

Tamamlanan ağaç: 300/300 | RAM: 1.57 GB


Model kaydediliyor: heavy_random_forest.joblib
Model kaydı tamamlandı: 1.8 sn | 67.54 MB
diagnostic_threshold                              0.5
precision                                     0.35591
recall                                       0.691613
f1                                           0.469969
roc_auc                                      0.926141
pr_auc                                       0.604982
tn                                              81601
fp                                               3880
fn                                                956
tp                                               2144
positive_prediction_rate_at_050              0.068006
model_name                          HeavyRandomForest
model_role                                      heavy
device                                            CPU
split_version                               common_v2
preprocessing_version                   lazy_cache_v1
imbalance_method                   balanced_sub

## 6. XGBoost

Önce CUDA denenir. GPU yapılandırması başarısız olursa aynı parametrelerle CPU fallback uygulanır ve kullanılan cihaz kaydedilir.

In [7]:
xgb_negative = int((tree_data["y_train"] == 0).sum())
xgb_positive = int((tree_data["y_train"] == 1).sum())
xgb_scale_pos_weight = xgb_negative / xgb_positive

xgb_base_params = {
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "n_estimators": 2_000,
    "learning_rate": 0.03,
    "max_depth": 8,
    "min_child_weight": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "scale_pos_weight": xgb_scale_pos_weight,
    "tree_method": "hist",
    "early_stopping_rounds": 100,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

def train_xgboost_with_fallback():
    attempts = ["cuda", "cpu"] if PREFERRED_DEVICE.upper() == "GPU" else ["cpu"]
    last_error = None
    for device in attempts:
        params = xgb_base_params.copy()
        params["device"] = device
        candidate = XGBClassifier(**params)
        print(f"\nXGBoost eğitimi başlıyor — cihaz: {device.upper()}")
        ram_before_local = ram_gb()
        started_local = time.perf_counter()
        try:
            candidate.fit(
                tree_data["X_train"], tree_data["y_train"],
                eval_set=[(tree_data["X_validation"], tree_data["y_validation"])],
                verbose=50,
            )
            return (
                candidate, device.upper(), time.perf_counter() - started_local,
                ram_before_local, ram_gb(),
            )
        except Exception as error:
            last_error = error
            print(f"{device.upper()} başarısız: {str(error)[:800]}")
            if device != attempts[-1]:
                print("CPU fallback denenecek.")
    raise RuntimeError("XGBoost GPU ve CPU üzerinde eğitilemedi.") from last_error

(
    xgb_model, xgb_device, xgb_training_seconds,
    xgb_ram_before, xgb_ram_after,
) = train_xgboost_with_fallback()

started = time.perf_counter()
xgb_probabilities = xgb_model.predict_proba(tree_data["X_validation"])[:, 1]
xgb_validation_inference = time.perf_counter() - started

xgb_metrics = finish_evaluation(
    model_name="XGBoost",
    model=xgb_model,
    data=tree_data,
    probabilities=xgb_probabilities,
    training_seconds=xgb_training_seconds,
    validation_inference_seconds=xgb_validation_inference,
    model_path=MODEL_DIR / "xgboost_heavy.joblib",
    device=xgb_device,
    imbalance_method="scale_pos_weight",
    ram_before=xgb_ram_before,
    ram_after=xgb_ram_after,
)


XGBoost eğitimi başlıyor — cihaz: CUDA
[0]	validation_0-aucpr:0.40133
[50]	validation_0-aucpr:0.54066
[100]	validation_0-aucpr:0.59123
[150]	validation_0-aucpr:0.61925
[200]	validation_0-aucpr:0.63533
[250]	validation_0-aucpr:0.64664
[300]	validation_0-aucpr:0.65564
[350]	validation_0-aucpr:0.66334
[400]	validation_0-aucpr:0.67047
[450]	validation_0-aucpr:0.67697
[500]	validation_0-aucpr:0.68284
[550]	validation_0-aucpr:0.68838
[600]	validation_0-aucpr:0.69296
[650]	validation_0-aucpr:0.69792
[700]	validation_0-aucpr:0.70283
[750]	validation_0-aucpr:0.70798
[800]	validation_0-aucpr:0.71161
[850]	validation_0-aucpr:0.71560
[900]	validation_0-aucpr:0.71882
[950]	validation_0-aucpr:0.72223
[1000]	validation_0-aucpr:0.72528
[1050]	validation_0-aucpr:0.72827
[1100]	validation_0-aucpr:0.73152
[1150]	validation_0-aucpr:0.73423
[1200]	validation_0-aucpr:0.73739
[1250]	validation_0-aucpr:0.74020
[1300]	validation_0-aucpr:0.74246
[1350]	validation_0-aucpr:0.74492
[1400]	validation_0-aucpr:0.746

c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\xgboost\core.py:751: UserWarning: [21:48:04] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Model kaydediliyor: xgboost_heavy.joblib
Model kaydı tamamlandı: 0.1 sn | 3.64 MB
diagnostic_threshold                            0.5
precision                                  0.434275
recall                                     0.828065
f1                                         0.569748
roc_auc                                    0.963524
pr_auc                                     0.770168
tn                                            82137
fp                                             3344
fn                                              533
tp                                             2567
positive_prediction_rate_at_050             0.06673
model_name                                  XGBoost
model_role                                    heavy
device                                         CUDA
split_version                             common_v2
preprocessing_version                 lazy_cache_v1
imbalance_method                   scale_pos_weight
threshold_selected                

## 7. Tree cache'ini bellekten çıkarma

In [8]:
del tree_data, heavy_rf_model, xgb_model
gc.collect()
print("Tree cache ve modeller bellekten çıkarıldı.")
print("RAM:", f"{ram_gb():.2f} GB")

Tree cache ve modeller bellekten çıkarıldı.
RAM: 0.44 GB


## 8. Heavy LightGBM

LightGBM native kategorik cache'i yüklenir. Önce GPU denenir; paket GPU desteği içermiyorsa CPU fallback uygulanır.

In [9]:
lightgbm_data = get_or_create_profile_cache(
    "lightgbm", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=False,
)
assert "X_test" not in lightgbm_data
assert np.array_equal(validation_ids, lightgbm_data["id_validation"])
assert np.array_equal(validation_target, lightgbm_data["y_validation"])

lgb_negative = int((lightgbm_data["y_train"] == 0).sum())
lgb_positive = int((lightgbm_data["y_train"] == 1).sum())
lgb_scale_pos_weight = lgb_negative / lgb_positive

lgb_base_params = {
    "objective": "binary",
    "n_estimators": 2_000,
    "learning_rate": 0.03,
    "num_leaves": 127,
    "max_depth": -1,
    "min_child_samples": 50,
    "subsample": 0.8,
    "subsample_freq": 1,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 0.5,
    "scale_pos_weight": lgb_scale_pos_weight,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbosity": -1,
}

def train_lightgbm_with_fallback():
    attempts = ["gpu", "cpu"] if PREFERRED_DEVICE.upper() == "GPU" else ["cpu"]
    last_error = None
    for device in attempts:
        params = lgb_base_params.copy()
        params["device_type"] = device
        candidate = LGBMClassifier(**params)
        print(f"\nLightGBM eğitimi başlıyor — cihaz: {device.upper()}")
        ram_before_local = ram_gb()
        started_local = time.perf_counter()
        try:
            candidate.fit(
                lightgbm_data["X_train"], lightgbm_data["y_train"],
                eval_set=[(lightgbm_data["X_validation"], lightgbm_data["y_validation"])],
                eval_metric="auc",
                callbacks=[lgb.early_stopping(100, verbose=True), lgb.log_evaluation(50)],
            )
            return (
                candidate, device.upper(), time.perf_counter() - started_local,
                ram_before_local, ram_gb(),
            )
        except Exception as error:
            last_error = error
            print(f"{device.upper()} başarısız: {str(error)[:800]}")
            if device != attempts[-1]:
                print("CPU fallback denenecek.")
    raise RuntimeError("LightGBM GPU ve CPU üzerinde eğitilemedi.") from last_error

(
    heavy_lgbm_model, lgbm_device, lgbm_training_seconds,
    lgbm_ram_before, lgbm_ram_after,
) = train_lightgbm_with_fallback()

started = time.perf_counter()
heavy_lgbm_probabilities = heavy_lgbm_model.predict_proba(
    lightgbm_data["X_validation"]
)[:, 1]
lgbm_validation_inference = time.perf_counter() - started

heavy_lgbm_metrics = finish_evaluation(
    model_name="HeavyLightGBM",
    model=heavy_lgbm_model,
    data=lightgbm_data,
    probabilities=heavy_lgbm_probabilities,
    training_seconds=lgbm_training_seconds,
    validation_inference_seconds=lgbm_validation_inference,
    model_path=MODEL_DIR / "lightgbm_heavy.joblib",
    device=lgbm_device,
    imbalance_method="scale_pos_weight",
    ram_before=lgbm_ram_before,
    ram_after=lgbm_ram_after,
)

lightgbm cache hazır; yeniden preprocessing yapılmayacak.

LightGBM eğitimi başlıyor — cihaz: GPU


c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


GPU başarısız: GPU Tree Learner was not enabled in this build.
Please recompile with CMake option -DUSE_GPU=1
CPU fallback denenecek.

LightGBM eğitimi başlıyor — cihaz: CPU


c:\Users\pc\anaconda3\envs\torchcuda\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Training until validation scores don't improve for 100 rounds
[50]	valid_0's auc: 0.946342	valid_0's binary_logloss: 0.167897
[100]	valid_0's auc: 0.956283	valid_0's binary_logloss: 0.164215
Early stopping, best iteration is:
[6]	valid_0's auc: 0.922262	valid_0's binary_logloss: 0.130364


Model kaydediliyor: lightgbm_heavy.joblib
Model kaydı tamamlandı: 0.0 sn | 0.19 MB
diagnostic_threshold                            0.5
precision                                       0.0
recall                                          0.0
f1                                              0.0
roc_auc                                    0.922262
pr_auc                                     0.506906
tn                                            85481
fp                                                0
fn                                             3100
tp                                                0
positive_prediction_rate_at_050                 0.0
model_name                            HeavyLightGBM
model_role                                    heavy
device                                          CPU
split_version                             common_v2
preprocessing_version                 lazy_cache_v1
imbalance_method                   scale_pos_weight
threshold_selected               

In [10]:
del lightgbm_data, heavy_lgbm_model
gc.collect()
print("LightGBM cache ve model bellekten çıkarıldı. RAM:", f"{ram_gb():.2f} GB")

LightGBM cache ve model bellekten çıkarıldı. RAM: 3.23 GB


## 9. Heavy CatBoost

CatBoost native kategorik cache'i yüklenir. Kategorik kolonlar cache dtype'larından tespit edilir. Önce GPU, hata durumunda CPU denenir.

In [11]:
catboost_data = get_or_create_profile_cache(
    "catboost", PROJECT_ROOT,
    quick_mode=QUICK_MODE, include_test=False,
)
assert "X_test" not in catboost_data
assert np.array_equal(validation_ids, catboost_data["id_validation"])
assert np.array_equal(validation_target, catboost_data["y_validation"])

catboost_categorical_columns = catboost_data["X_train"].select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()
print("CatBoost kategorik kolon sayısı:", len(catboost_categorical_columns))

catboost_base_params = {
    "iterations": 2_000,
    "depth": 8,
    "learning_rate": 0.03,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "l2_leaf_reg": 5.0,
    "random_strength": 1.0,
    "use_best_model": True,
    "allow_writing_files": False,
    "verbose": 50,
}

def train_catboost_with_fallback():
    attempts = ["GPU", "CPU"] if PREFERRED_DEVICE.upper() == "GPU" else ["CPU"]
    last_error = None
    for device in attempts:
        params = catboost_base_params.copy()
        params["task_type"] = device
        if device == "GPU":
            params["devices"] = "0"
        else:
            params["thread_count"] = -1
        candidate = CatBoostClassifier(**params)
        print(f"\nCatBoost eğitimi başlıyor — cihaz: {device}")
        ram_before_local = ram_gb()
        started_local = time.perf_counter()
        try:
            candidate.fit(
                catboost_data["X_train"], catboost_data["y_train"],
                cat_features=catboost_categorical_columns,
                eval_set=(catboost_data["X_validation"], catboost_data["y_validation"]),
                early_stopping_rounds=100,
            )
            return (
                candidate, device, time.perf_counter() - started_local,
                ram_before_local, ram_gb(),
            )
        except Exception as error:
            last_error = error
            print(f"{device} başarısız: {str(error)[:800]}")
            if device != attempts[-1]:
                print("CPU fallback denenecek.")
    raise RuntimeError("CatBoost GPU ve CPU üzerinde eğitilemedi.") from last_error

(
    heavy_catboost_model, catboost_device, catboost_training_seconds,
    catboost_ram_before, catboost_ram_after,
) = train_catboost_with_fallback()

started = time.perf_counter()
heavy_catboost_probabilities = heavy_catboost_model.predict_proba(
    catboost_data["X_validation"]
)[:, 1]
catboost_validation_inference = time.perf_counter() - started

heavy_catboost_metrics = finish_evaluation(
    model_name="HeavyCatBoost",
    model=heavy_catboost_model,
    data=catboost_data,
    probabilities=heavy_catboost_probabilities,
    training_seconds=catboost_training_seconds,
    validation_inference_seconds=catboost_validation_inference,
    model_path=MODEL_DIR / "catboost_heavy.joblib",
    device=catboost_device,
    imbalance_method="auto_class_weights_balanced",
    ram_before=catboost_ram_before,
    ram_after=catboost_ram_after,
)

catboost cache bulunamadı; ortak Parquet'ten oluşturuluyor.

BAŞLADI: Ortak Parquet cache'ini yükleme
TAMAMLANDI: Ortak Parquet cache'ini yükleme | geçen süre: 0.4 sn | RAM: 5.28 GB

BAŞLADI: catboost/train Parquet kaydı
TAMAMLANDI: catboost/train Parquet kaydı | geçen süre: 3.5 sn | RAM: 4.21 GB

BAŞLADI: catboost/validation Parquet kaydı
TAMAMLANDI: catboost/validation Parquet kaydı | geçen süre: 0.8 sn | RAM: 3.00 GB

BAŞLADI: catboost/test Parquet kaydı
TAMAMLANDI: catboost/test Parquet kaydı | geçen süre: 0.8 sn | RAM: 2.98 GB
CatBoost kategorik kolon sayısı: 42

CatBoost eğitimi başlıyor — cihaz: GPU


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.8641269	best: 0.8641269 (0)	total: 224ms	remaining: 7m 27s
50:	test: 0.9225657	best: 0.9225657 (50)	total: 5.89s	remaining: 3m 45s
100:	test: 0.9433620	best: 0.9433620 (100)	total: 11s	remaining: 3m 26s
150:	test: 0.9512838	best: 0.9512838 (150)	total: 16.3s	remaining: 3m 19s
200:	test: 0.9564996	best: 0.9565263 (199)	total: 21.5s	remaining: 3m 12s
250:	test: 0.9591617	best: 0.9591617 (250)	total: 26.7s	remaining: 3m 6s
300:	test: 0.9613683	best: 0.9613683 (300)	total: 31.9s	remaining: 2m 59s
350:	test: 0.9634399	best: 0.9634399 (350)	total: 37.1s	remaining: 2m 54s
400:	test: 0.9650817	best: 0.9650817 (400)	total: 42.5s	remaining: 2m 49s
450:	test: 0.9662704	best: 0.9662704 (450)	total: 47.9s	remaining: 2m 44s
500:	test: 0.9671487	best: 0.9671487 (500)	total: 53.1s	remaining: 2m 38s
550:	test: 0.9681514	best: 0.9681543 (549)	total: 58.3s	remaining: 2m 33s
600:	test: 0.9687892	best: 0.9687892 (600)	total: 1m 3s	remaining: 2m 27s
650:	test: 0.9693938	best: 0.9693938 (650)	tota

Model kaydediliyor: catboost_heavy.joblib
Model kaydı tamamlandı: 3.7 sn | 139.60 MB
diagnostic_threshold                                       0.5
precision                                             0.458529
recall                                                0.880968
f1                                                    0.603136
roc_auc                                               0.974278
pr_auc                                                0.834212
tn                                                       82256
fp                                                        3225
fn                                                         369
tp                                                        2731
positive_prediction_rate_at_050                       0.067238
model_name                                       HeavyCatBoost
model_role                                               heavy
device                                                     GPU
split_version                    

In [12]:
del catboost_data, heavy_catboost_model
gc.collect()
print("CatBoost cache ve model bellekten çıkarıldı. RAM:", f"{ram_gb():.2f} GB")

CatBoost cache ve model bellekten çıkarıldı. RAM: 4.11 GB


## 10. Validation karşılaştırması ve olasılık kaydı

PR-AUC/ROC-AUC eşikten bağımsızdır. `0.50` precision/recall/F1 değerleri yalnızca tanısaldır. Kesin ağır model ve threshold seçimi yapılmaz.

In [13]:
comparison = pd.DataFrame([
    heavy_rf_metrics,
    xgb_metrics,
    heavy_lgbm_metrics,
    heavy_catboost_metrics,
]).sort_values(["pr_auc", "roc_auc"], ascending=False).reset_index(drop=True)

comparison_path = PATHS.metrics / "heavy_model_comparison_validation.csv"
comparison.to_csv(comparison_path, index=False)

validation_predictions = pd.DataFrame({
    "TransactionID": validation_ids,
    "y_true": validation_target,
    "heavy_random_forest_probability": heavy_rf_probabilities,
    "xgboost_probability": xgb_probabilities,
    "heavy_lightgbm_probability": heavy_lgbm_probabilities,
    "heavy_catboost_probability": heavy_catboost_probabilities,
})
predictions_path = PREDICTIONS_DIR / "heavy_models_validation_predictions.parquet"
validation_predictions.to_parquet(predictions_path, index=False, compression="zstd")

metadata = {
    "experiment_stage": "task_3_heavy_model_comparison",
    "split_version": "common_v2",
    "test_used": False,
    "threshold_tuned": False,
    "diagnostic_threshold": DIAGNOSTIC_THRESHOLD,
    "models": comparison.to_dict(orient="records"),
}
metadata_path = PATHS.metadata / "heavy_model_comparison_metadata.json"
with metadata_path.open("w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

display(comparison)
print("Karşılaştırma:", comparison_path.relative_to(PROJECT_ROOT))
print("Validation olasılıkları:", predictions_path.relative_to(PROJECT_ROOT))
print("Metadata:", metadata_path.relative_to(PROJECT_ROOT))

,diagnostic_threshold,precision,recall,f1,roc_auc,pr_auc,tn,fp,fn,tp,...,imbalance_method,threshold_selected,training_seconds,validation_inference_seconds,inference_seconds_10k_mean,inference_seconds_10k_std,inference_sample_rows,model_size_mb,ram_before_gb,ram_after_fit_gb
0,0.5,0.458529,0.880968,0.603136,0.974278,0.834212,82256,3225,369,2731,...,auto_class_weights_balanced,False,184.584949,1.012581,0.124249,0.002607,10000,139.600267,4.090561,4.568565
1,0.5,0.434275,0.828065,0.569748,0.963524,0.770168,82137,3344,533,2567,...,scale_pos_weight,False,37.662317,0.571493,0.066894,0.000910,10000,3.636289,1.577713,1.798904
2,0.5,0.355910,0.691613,0.469969,0.926141,0.604982,81601,3880,956,2144,...,balanced_subsample,False,259.393200,0.793163,0.094810,0.001890,10000,67.544916,1.385845,1.572678
3,0.5,0.000000,0.000000,0.000000,0.922262,0.506906,85481,0,3100,0,...,scale_pos_weight,False,7.832314,0.109417,0.038215,0.012453,10000,0.185541,3.594913,3.498871


Karşılaştırma: outputs\metrics\heavy_model_comparison_validation.csv
Validation olasılıkları: outputs\predictions\heavy_models_validation_predictions.parquet
Metadata: outputs\metadata\heavy_model_comparison_metadata.json


# Görev 3 tamamlanma koşulları

- Dört ağır model aynı ortak train/validation splitinde çalıştırıldı.
- Test spliti yüklenmedi veya değerlendirilmedi.
- Threshold seçilmedi; `0.50` yalnızca tanısal karşılaştırma için kullanıldı.
- ROC-AUC, PR-AUC, precision, recall, F1, FP/FN, süre, RAM, cihaz ve model boyutu kaydedildi.
- GPU destekleyen modellerde GPU denendi ve hata durumunda CPU fallback uygulandı.
- Validation olasılıkları Görev 4 threshold analizi için saklandı.

Sonraki görev, onaydan sonra validation threshold tuning aşamasıdır.